We use Custom model best.pt

EasyOCR for the character extraction/reading

OpenCV (cv2) enhancing copped box immage quality

Padding Logic to increase each crop by 15 pixels on each side to ensure shoortened crops.

Strict Allowlist using all aplhabets numbers and . or - only

Sorting Logic: reads from left to right of the image


In [ ]:

# CELL 1: SETUP

import os
import torch

print(" Installing dependencies...")
!pip install -q ultralytics easyocr opencv-python-headless matplotlib

print(f" Ready. GPU Available: {torch.cuda.is_available()}")


In [ ]:

# CELL 2: V7 PIPELINE (Custom Model + Precision OCR)


import os
import cv2
import numpy as np
import torch
import easyocr
from ultralytics import YOLO

# CONFIGURATION
IMAGE_PATH = "IMG_1992.jpeg"
MODEL_PATH = "super_model_v1.pt"           # our custom model
OUTPUT_DIR = "/content/v7_results"
CONFIDENCE_THRESHOLD = 0.4       # Only keep confident detections
PADDING = 15                     # Pixels to add around the box

# Initialize EasyOCR
reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())

def preprocess_for_ocr(image):
    """
    Standardizes the sticker image for maximum readability.
    """
    # 1. Upscale (3x)  Critical for small decimals like .3
    zoom = cv2.resize(image, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

    # 2. Grayscale
    gray = cv2.cvtColor(zoom, cv2.COLOR_BGR2GRAY)

    # 3. CLAHE (Contrast Boost)  Makes black text pop on white paper
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)

    # 4. Sharpening - Crispens edges of letters
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharp = cv2.filter2D(enhanced, -1, kernel)

    return sharp

def sort_boxes(boxes):
    """
    Sorts detections in reading order: Top-to-Bottom (Shelves), then Left-to-Right.
    """
    # Simple logic: Sort by Y (vertical) first with a tolerance, then X
    # Convert to list of [x1, y1, x2, y2, conf]
    if len(boxes) == 0: return []

    # Heuristic: Sort by Y center
    # If Y centers are within 50px, consider them on the same "shelf"
    boxes = sorted(boxes, key=lambda b: (int(b[1]/100), b[0]))
    return boxes

def main_v7():
    if not os.path.exists(IMAGE_PATH):
        print(f" Error: {IMAGE_PATH} not found.")
        return
    if not os.path.exists(MODEL_PATH):
        print(f" Error: {MODEL_PATH} not found. Please upload your trained model.")
        return

    os.makedirs(f"{OUTPUT_DIR}/crops", exist_ok=True)

    print(f" Loading {MODEL_PATH}...")
    model = YOLO(MODEL_PATH)

    print(f" Detecting stickers in {IMAGE_PATH}...")
    img = cv2.imread(IMAGE_PATH)
    h_img, w_img = img.shape[:2]

    # Run Inference
    results = model.predict(IMAGE_PATH, conf=CONFIDENCE_THRESHOLD, verbose=False)

    # Extract Boxes
    detections = []
    for r in results:
        for box in r.boxes:
            # Get coordinates
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf)
            detections.append([int(x1), int(y1), int(x2), int(y2), conf])

    # Sort Left-to-Right (Scanning order)
    detections = sort_boxes(detections)

    print(f" Found {len(detections)} stickers.")
    print("\n--- EXTRACTION RESULTS ---")

    with open(f"{OUTPUT_DIR}/final_report.txt", "w") as f:
        f.write(f"V7 RESULTS ({len(detections)} Stickers)\n================================\n")

        for i, (x1, y1, x2, y2, conf) in enumerate(detections):
            # --- 1. SAFE CROP ---
            # Add padding to ensure we capture the full "Z" and edges
            # But don't go outside image boundaries
            cx1 = max(0, x1 - PADDING)
            cy1 = max(0, y1 - PADDING)
            cx2 = min(w_img, x2 + PADDING)
            cy2 = min(h_img, y2 + PADDING)

            crop = img[cy1:cy2, cx1:cx2]

            # --- 2. ENHANCE ---
            clean_crop = preprocess_for_ocr(crop)

            # Save debug image
            filename = f"sticker_{i+1:03d}.jpg"
            cv2.imwrite(f"{OUTPUT_DIR}/crops/{filename}", clean_crop)

            # --- 3. READ ---
            # We allow specific characters common in library codes
            allowlist = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.- "
            text_results = reader.readtext(clean_crop, detail=1, allowlist=allowlist)

            # Sort lines top to bottom
            lines = []
            for (bbox, text, ocr_conf) in text_results:
                if ocr_conf > 0.2: # Low threshold to catch everything
                     # Use center Y to sort
                    y_center = (bbox[0][1] + bbox[2][1]) / 2
                    lines.append((y_center, text.upper()))

            lines.sort(key=lambda x: x[0])
            full_text = " | ".join([l[1] for l in lines])

            # Print
            print(f"Sticker {i+1:02d}: {full_text}")
            f.write(f"Sticker {i+1:02d}: {full_text}\n")

    os.system(f"zip -r v7_results.zip {OUTPUT_DIR}")
    print("\n Done. Download 'v7_results.zip'")

if __name__ == "__main__":
    main_v7()

Loading super_model_v1.pt...
 Detecting stickers in IMG_1992.jpeg...
 Found 24 stickers.

--- EXTRACTION RESULTS ---
Sticker 01: BL | 2747.3 | T67 | 2023
Sticker 02: 2 | BL | 2747.2 | A75 | 2003
Sticker 03: BL | 2747.8 | B35 | 2012
Sticker 04: BL | 82747.8 | .C75 | 2004
Sticker 05: BL | 2747.8 | C758 | 2012
Sticker 06: BL | 2747.8 | L3519 | 2011
Sticker 07: BL | 2750 | B8719 | 2013
Sticker 08: BL | 2765 | 9 | E85 | B8S | 2007
Sticker 09: BL | 2765 | F8 | B3819 | 2018
Sticker 10: BL | 2765 | F8 | F68 | 2016
Sticker 11: BL | 2765 | T9 | H33 | 11998
Sticker 12: BL | 2765 | T9 | L3S | 2001 | 8
Sticker 13: BL | 2773 | M4719 | 1995
Sticker 14: BL | 2773 | 098 | 2018
Sticker 15: BL | 2775 | 37 | 1958
Sticker 16: BL | 2775.3 | D39 | 2006
Sticker 17: BL | 2775.3 | D46 | 2006
Sticker 18: BL | 2775.3 | D3919 | 2013
Sticker 19: BL | 2775.3 | D46 | 2006 | 22 | C
Sticker 20: BL | 2775.3 | H37 | 2005
Sticker 21: BL | 2776 | D419 | 2012
Sticker 22: BL | 2780 | R87
Sticker 23: BM | 199 | S3 | S7 | 2016